# 项目 B 实验：沿一条预测走到文本增量

本实验始终研究同一个问题：市场特征已经给出排序后，模板公告是否提供额外信息？我们先运行固定基线，再沿一个实际月末追踪字段、标签、预测和仓位，最后保持同样的样本与交易规则比较模型。数据全部由本项目生成，默认不访问网络，也不加载语义模型。

一行样本由“某资产、某决策月末”确定。`future_excess_return` 是未来收益的合成响应，`synthetic_text_signal` 是生成器知道而研究者不知道的潜在状态；两者都不能进入预测特征。我们会在时间边界处计算标签什么时候成熟，避免把未来结果当作训练当天已知的数据。

In [ ]:
from pathlib import Path
import sys
import tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

candidates = [candidate for base in [Path.cwd(), *Path.cwd().parents]
              for candidate in [base, base / "research/projects/ml_llm_research"]]
PROJECT = next(p for p in candidates if (p / "src/evaluate.py").exists())
sys.path.insert(0, str(PROJECT))
from src.run import main as run_baseline
from src.models import split_dates, chronological_masks, fit_predict
from src.evaluate import rank_ic, top_bottom_returns, summarize

workspace = tempfile.TemporaryDirectory(prefix="research-text-")
LAB_OUTPUT = Path(workspace.name)
run_baseline(output_dir=LAB_OUTPUT)
panel = pd.read_csv(LAB_OUTPUT / "panel.csv", parse_dates=["decision_date", "published_at"])
print("Rows:", len(panel), "Assets:", panel["asset"].nunique(), "Months:", panel["decision_date"].nunique())
print(panel[["decision_date", "asset", "published_at", "text", "future_excess_return"]].head().to_string(index=False))

## 1. 一条公告怎样变成数值

先看 `panel.csv` 的一行。发布时间比决策早两天，文本在这里已经可见；词典函数将它转换成情绪、uncertainty 计数和长度。市场模型看五个市场字段，文本模型多看三个词典字段。`uncertainty_count` 只数词，不知道 “lower uncertainty” 是风险下降，所以读到一个数值还应回看原句。

默认生成器只有三种模板。下面对每一种只展示一行，是为了读懂变换，不代表三条文本足以验证语言理解能力。真实研究还要检查公司实体、修订、接收时间、去重与模型版本。当前所有文本都非空，因此覆盖率为 100%；这个简化会掩盖真实文本覆盖的选择性。

In [ ]:
text_columns = ["text", "lexical_sentiment", "uncertainty_count", "text_length"]
print(panel[text_columns].drop_duplicates().to_string(index=False))
print({"visible_before_decision": bool((panel["published_at"] < panel["decision_date"]).all()),
       "nonempty_text_rate": float(panel["text"].fillna("").str.strip().ne("").mean())})

## 2. 从决策日期切分，走到标签成熟切分

固定运行器按 60%/20%/20% 的决策日期切分，模型只在训练段拟合。这避免了随机混合过去与未来，但还没有完整回答“训练时是否拿得到标签”。假设一月末那一行标签描述二月收益，那么一月末当天还不能拿它训练。

这里显式设定每行标签在下一个月末成熟，并以训练段末日为训练快照：只有 `label_end <= train_end` 的训练行可用。验证段用于在其末日作选择，也只读取该日已成熟的验证标签。测试段从验证快照之后开始，最后在所有测试标签都已实现后评价。由于标签是一月期，边界各删除一个月；这一步叫清理跨边界标签，不是随意留空一段数据。

月末在真实交易中可能是假日，本实验只有月度时钟；真实适配必须进一步落到交易所日历、公开时间和最早成交价格。下面的表将决策日期与成熟日期放在一起，防止用列位置代替信息时间。

In [ ]:
train_end, validation_end = split_dates(panel)
panel["label_end"] = panel["decision_date"] + pd.offsets.MonthEnd(1)
original_masks = chronological_masks(panel, train_end, validation_end)
masks = {
    "train": original_masks["train"] & (panel["label_end"] <= train_end),
    "validation": original_masks["validation"] & (panel["label_end"] <= validation_end),
    "test": original_masks["test"],
}
split_rows = []
for name, mask in masks.items():
    subset = panel.loc[mask]
    split_rows.append({"split": name, "rows": len(subset),
                       "first_decision": subset["decision_date"].min().date(),
                       "last_decision": subset["decision_date"].max().date(),
                       "last_label_end": subset["label_end"].max().date()})
print(pd.DataFrame(split_rows).to_string(index=False))

## 3. 在同一批样本上重建三个预测器

现在冻结这套边界，比较训练均值、市场岭回归、市场加词典岭回归。源码 `fit_predict` 每次只用传入的训练样本计算均值、标准差与系数，因此验证和测试特征不会反向影响预处理。为使本实验只回答“文本是否有增量”，惩罚强度保持源码默认值，不在测试结果出来后选择新的窗口或词典。

均值预测器对同一个月所有资产给出相同预测，可以用于均方误差比较，但没有截面排序信息。rank IC 是同一日期中预测排名与标签排名的相关，再跨日期平均；当预测恒定时没有可定义的相关。源码把这种情况汇总为零，这是接口约定，不能解释成已经测得一个精确为零的相关系数。下面不将它的同分排序组合当作策略，而用现金零仓位作交易参照。

In [ ]:
market_columns = ["momentum", "volatility", "volume_change", "beta", "market_return"]
specifications = {
    "mean": (market_columns, "mean"),
    "market": (market_columns, "ridge"),
    "market_text": (market_columns + ["lexical_sentiment", "uncertainty_count", "text_length"], "ridge"),
}
train = panel.loc[masks["train"]].copy()
prediction_frames = []
for split_name in ["validation", "test"]:
    evaluation = panel.loc[masks[split_name]].copy()
    for name, (columns, method) in specifications.items():
        frame = evaluation[["decision_date", "asset", "future_excess_return"]].copy()
        frame["prediction"] = fit_predict(train, evaluation, columns, method)
        frame["model"], frame["split"] = name, split_name
        prediction_frames.append(frame)
predictions = pd.concat(prediction_frames, ignore_index=True)
rows = []
for (split_name, model), group in predictions.groupby(["split", "model"]):
    rows.append({"split": split_name, "model": model,
                 "MSE": ((group["prediction"] - group["future_excess_return"]) ** 2).mean(),
                 "rank_IC": rank_ic(group) if model != "mean" else np.nan})
print(pd.DataFrame(rows).round(5).to_string(index=False))

读表先比较相同 split 的行。若文本只在训练数据好、验证和测试均无改善，不能说它提供稳定增量；若验证有改善而测试没有，先保留负结果，再检查抽样误差、覆盖或机制变化。由于本例模板信息来自已知生成器，最终结论只适用于这组教学机制。

验证与测试只有二十余个月，各月十二只资产，几百行不能当成几百次独立市场实验。按月共同变化会产生截面依赖；需要推断时，应在日期层比较 IC 或组合差，必要时保留时间依赖做块重抽样，不能把每个资产标签都独立打乱。

## 4. 选一个实际测试月，把预测变成权重

回到开头的三资产账本，这次使用真实生成的十二只教学资产。取文本模型第一个测试月，最高四只每只 +0.25，最低四只每只 -0.25，中间四只为零。总多头为 1、总空头为 -1，因此净敞口为零，总绝对敞口为 2。

逐资产贡献是权重乘未来标签，求和应等于四只多头标签均值减四只空头均值。此时刚从空仓建仓，换手为权重绝对值之和 2；按单边 10 基点，成本为 20 基点。下面只把已经算好的测试预测拿来做账，不再回头调整模型。

In [ ]:
test_text = predictions.loc[(predictions["split"] == "test") & (predictions["model"] == "market_text")].copy()
first_date = test_text["decision_date"].min()
ledger = test_text.loc[test_text["decision_date"] == first_date].sort_values("prediction").copy()
count = max(1, len(ledger) // 3)
ledger["weight"] = 0.0
ledger.loc[ledger.index[:count], "weight"] = -1 / count
ledger.loc[ledger.index[-count:], "weight"] = 1 / count
ledger["gross_contribution"] = ledger["weight"] * ledger["future_excess_return"]
initial_turnover = ledger["weight"].abs().sum()
print(ledger[["asset", "prediction", "future_excess_return", "weight", "gross_contribution"]].round(5).to_string(index=False))
print({"net_exposure": float(ledger["weight"].sum()),
       "gross_exposure": float(ledger["weight"].abs().sum()),
       "gross": float(ledger["gross_contribution"].sum()),
       "initial_cost": float(initial_turnover * 10 / 10000)})

目标权重差近似默认不处理月内价格造成的漂移。例如本月 +0.25 的多头涨了 10%，下月维持 +0.25 仍要卖出一些份额；源码若两月目标一样会记作零换手。因此本项目练习的是可解释的目标组合比较，还不是完整的股票和现金自融资账本。第一、二、六章已经给出真实份额、现金与成交的做法，真实数据扩展应将两者连接。

此外，净敞口为零不等于市场 beta 为零，也不等于没有融资和借券风险。若多头高 beta、空头低 beta，组合仍可能偏向市场。后面应同时查看暴露、容量和借券成本，而不能只看月度净值。

## 5. 固定预测，单独改变成本

成本压力实验不需要重新训练。对同一组预测按 0、10、25、50 基点收费，差异只能来自换手乘成本差。平均换手越高，平均净收益随成本上升下降得越快；IC 完全不受这种重新收费影响，因为预测和标签没有改变。这正好解释为什么一个 IC 更高的文本模型，仍可能没有净收益增量。

现金比较的收益和交易量均为零。它不是均值模型强行排序后的组合，也不涉及为得到“零收益”挑选资产。

In [ ]:
rows = []
for model in ["market", "market_text"]:
    group = predictions.loc[(predictions["split"] == "test") & (predictions["model"] == model)]
    for cost_bps in [0, 10, 25, 50]:
        portfolio = top_bottom_returns(group, cost_bps=cost_bps)
        rows.append({"model": model, "cost_bps": cost_bps,
                     "average_net": portfolio["net"].mean(),
                     "turnover": portfolio["turnover"].mean()})
cost_table = pd.DataFrame(rows)
print(cost_table.pivot(index="model", columns="cost_bps", values="average_net").round(5))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for model in ["market", "market_text"]:
    group = predictions.loc[(predictions["split"] == "test") & (predictions["model"] == model)]
    portfolio = top_bottom_returns(group, cost_bps=10)
    axes[0].plot(portfolio["decision_date"], (1 + portfolio["net"]).cumprod(), label=model)
    sensitivity = cost_table.loc[cost_table["model"] == model]
    axes[1].plot(sensitivity["cost_bps"], sensitivity["average_net"], "o-", label=model)
axes[0].axhline(1, color="grey", linestyle="--", label="cash")
axes[1].axhline(0, color="grey", linestyle="--")
axes[0].set(title="Frozen test predictions: net equity", ylabel="equity")
axes[1].set(title="Cost sensitivity", xlabel="one-way cost (bp)", ylabel="mean monthly net return")
for ax in axes:
    ax.legend()
fig.tight_layout()
plt.show()

## 6. 手工决策与参考答案

### 问题 A：从一行标签判断能否训练

一月末的标签覆盖二月，模型在一月末拟合、二月末第一次做验证。是否可以因为一月行属于 train 就读取该标签？

<details><summary>参考答案</summary>

不可以。行的决策日属于 train，但标签到二月末才成熟，模型拟合当日仍不知道。应记录标签结束和可获得时间，并删除跨过训练快照的标签；若未来标签来自修订或延迟披露，还应进一步推迟可用时间。本实验每月一期，因此去掉最后一个训练月。

</details>

### 问题 B：一次翻转要交易多少

某只资产上一期权重 +0.25，下一期为 -0.25，账户规模暂按不变处理，单边成本 20 基点。只算这只资产的换手与成本是多少？

<details><summary>参考答案</summary>

需要先卖掉 0.25 多头，再建立 0.25 空头，交易量为 0.5，成本为 \(0.5\times20\times10^{-4}=0.1\%\)。这仍是固定权重近似；真实交易量要从已有份额和成交前净值求得。全组合其他资产的交易还需另外累加。

</details>

### 问题 C：文本理解和收益预测怎样分开

FinBERT 比词典更准确地区分 “uncertainty increased” 与 “uncertainty decreased”，但样本外净收益没有提高。下一步先做什么？

<details><summary>参考答案</summary>

先保留两个结论：文本分类有所改善，当前交易实验未发现净收益增量。然后在共同样本上查看增量是否已经被市场特征解释，预测提升是否集中在少数月份，是否被换手和成本消耗。若只在模板数据上评价，还需要有时间版本和人工标签的新事件数据；不应仅因语义模型更复杂而继续在测试集上调参。

</details>

## 7. 把同一条链写进报告

报告从本次严格切分表和一个月的账本开始：哪些日期用于拟合，哪些标签在当时成熟，文本多提供哪些字段，预测如何变成 +1/-1 的多空组合，成本在哪一步扣除。然后比较市场与文本的验证、测试和成本压力；最后解释负结果和真实扩展所缺的证据。

语义模型是可选的特征替换，不是自动买卖指令。若要运行，按项目 README 准备本地模型，记录版本与失败样本，仍使用这里的共同样本、标签成熟边界和交易评价。不要在看过测试结果之后，再把最有利的模型、成本或月份定义为原先的研究问题。